In [ ]:
# === KONFIGURACJA ===
# Token Kaggle, potrzebny do pobrania datasetu dla Grad-CAM
KAGGLE_USERNAME = 'USERNAME'
KAGGLE_KEY = 'TOKEN-KEY'

DRIVE_BASE = '/content/drive/MyDrive/plant_disease_classification'
RESULTS_DIR = f'{DRIVE_BASE}/results'
OUTPUT_DIR = f'{DRIVE_BASE}/aggregate'

# Lokalna ścieżka do datasetu (do Grad-CAM)
LOCAL_DATA = '/content/plantvillage'
SRC = f'{LOCAL_DATA}/PlantVillage'

FOLD_FOR_GRADCAM = 0  # Podział do wizualizacji Grad-CAM (0-4)


In [ ]:
# === MONTOWANIE DRIVE ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Drive zamontowany, wyniki agregacji w: {OUTPUT_DIR}')


In [ ]:
# === POBIERANIE DATASETU ===
if not os.path.exists(SRC):
    if KAGGLE_USERNAME == 'TWOJA_NAZWA_KAGGLE' or KAGGLE_KEY == 'WKLEJ_TOKEN_TUTAJ':
        raise ValueError(
            'Wklej swoj username i token Kaggle w komorce 2. '
            'Token: https://www.kaggle.com/settings/account -> API -> Create New Token'
        )

    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY'] = KAGGLE_KEY

    print('Pobieram dataset PlantVillage (potrzebny do Grad-CAM)...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    ret = os.system(f'kaggle datasets download -d emmarex/plantdisease -p {LOCAL_DATA} --unzip --quiet')
    if ret != 0:
        raise RuntimeError('Pobieranie nieudane. Sprawdz username i token.')

print(f'Dataset: {SRC}')


In [ ]:
# === SPRAWDZENIE OBECNOŚCI WYNIKÓW WSZYSTKICH PODZIAŁÓW ===
import json

fold_dirs = {}
for fold_idx in range(5):
    fd = f'{RESULTS_DIR}/fold_{fold_idx}'
    done_marker = f'{fd}/fold_{fold_idx}_done.json'
    if os.path.exists(done_marker):
        fold_dirs[fold_idx] = fd
        print(f'Podział {fold_idx+1}: {fd}')
    else:
        print(f'Podział {fold_idx+1}: BRAK (nie znaleziono {done_marker})')

assert len(fold_dirs) == 5, f'Znaleziono tylko {len(fold_dirs)} z 5 podziałów'


In [ ]:
# === IMPORTY ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon


In [ ]:
# === WCZYTANIE WYNIKÓW WSZYSTKICH PODZIAŁÓW ===
all_results = {'ResNet50': [], 'VGG16': []}
all_per_class = {'ResNet50': [], 'VGG16': []}
all_cms = {'ResNet50': [], 'VGG16': []}
all_histories = {'ResNet50': [], 'VGG16': []}

for fold_idx in range(5):
    fold_dir = fold_dirs[fold_idx]
    for model_name, fname in [('ResNet50', 'resnet50'), ('VGG16', 'vgg16')]:
        with open(f'{fold_dir}/{fname}_metrics.json') as f:
            data = json.load(f)
        all_results[model_name].append(data)
        all_per_class[model_name].append(data['per_class'])
        all_histories[model_name].append({
            'phase1': data['history_phase1'],
            'phase2': data['history_phase2'],
        })

        preds = np.load(f'{fold_dir}/{fname}_predictions.npz', allow_pickle=True)
        all_cms[model_name].append(preds['confusion_matrix'])

class_names = all_results['ResNet50'][0]['class_names']
print(f'Wczytano: {len(all_results["ResNet50"])} podziałów po 2 modele')
print(f'Klasy: {class_names}')


In [ ]:
# === ŚREDNIE WARTOŚCI METRYK ===
metrics_to_report = ['test_accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted']
metric_labels = {
    'test_accuracy': 'Dokładność',
    'balanced_accuracy': 'Dokładność zbalansowana',
    'f1_macro': 'F1 makro',
    'f1_weighted': 'F1 ważone',
}

table2_rows = []
for metric in metrics_to_report:
    resnet_vals = np.array([r[metric] for r in all_results['ResNet50']])
    vgg_vals = np.array([r[metric] for r in all_results['VGG16']])

    if not np.allclose(resnet_vals, vgg_vals):
        stat, p_value = wilcoxon(resnet_vals, vgg_vals)
        p_str = f'{p_value:.4f}'
    else:
        p_str = 'identyczne'

    table2_rows.append({
        'Metryka': metric_labels[metric],
        'ResNet50 (średnia, odchylenie)': f'{resnet_vals.mean()*100:.2f}% ± {resnet_vals.std()*100:.2f}%',
        'VGG16 (średnia, odchylenie)': f'{vgg_vals.mean()*100:.2f}% ± {vgg_vals.std()*100:.2f}%',
        'ResNet50 (kolejne podziały)': ', '.join(f'{v*100:.2f}%' for v in resnet_vals),
        'VGG16 (kolejne podziały)': ', '.join(f'{v*100:.2f}%' for v in vgg_vals),
        'p-wartość testu Wilcoxona': p_str,
    })

table2 = pd.DataFrame(table2_rows)
print(table2.to_string(index=False))
table2.to_csv(f'{OUTPUT_DIR}/table2_summary.csv', index=False, encoding='utf-8-sig')


In [ ]:
# === INTERPRETACJA TESTU WILCOXONA ===
print('=' * 60)
print('INTERPRETACJA TESTU WILCOXONA')
print('=' * 60)

for metric in metrics_to_report:
    resnet_vals = np.array([r[metric] for r in all_results['ResNet50']])
    vgg_vals = np.array([r[metric] for r in all_results['VGG16']])

    if np.allclose(resnet_vals, vgg_vals):
        print(f'{metric_labels[metric]:>25}: wartości identyczne, test nieaplikowalny')
        continue

    stat, p = wilcoxon(resnet_vals, vgg_vals)
    diff = (vgg_vals.mean() - resnet_vals.mean()) * 100
    direction = 'VGG16 > ResNet50' if diff > 0 else 'ResNet50 > VGG16'

    if p < 0.05:
        verdict = f'ISTOTNE STATYSTYCZNIE (p={p:.4f}, alfa=0.05)'
    else:
        verdict = f'brak istotności statystycznej (p={p:.4f})'

    print(f'{metric_labels[metric]:>25}: różnica {diff:+.2f}pp ({direction})')
    print(f'{"":>25}  {verdict}')


In [ ]:
# === PRECYZJA I CZUŁOŚĆ DLA KAŻDEJ KLASY ===
def aggregate_per_class(per_class_list, metric='precision'):
    values = {cls: [] for cls in class_names}
    for fold_data in per_class_list:
        for cls in class_names:
            values[cls].append(fold_data[cls][metric])
    return {cls: (np.mean(vs), np.std(vs)) for cls, vs in values.items()}

table3_rows = []
for cls in class_names:
    row = {'Klasa': cls}
    for model in ['ResNet50', 'VGG16']:
        prec = aggregate_per_class(all_per_class[model], 'precision')[cls]
        rec = aggregate_per_class(all_per_class[model], 'recall')[cls]
        row[f'Precyzja {model}'] = f'{prec[0]:.3f} ± {prec[1]:.3f}'
        row[f'Czułość {model}'] = f'{rec[0]:.3f} ± {rec[1]:.3f}'
    table3_rows.append(row)

table3 = pd.DataFrame(table3_rows)
print(table3.to_string(index=False))
table3.to_csv(f'{OUTPUT_DIR}/table3_per_class.csv', index=False, encoding='utf-8-sig')


In [ ]:
# === KRZYWE UCZENIA ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, model in enumerate(['ResNet50', 'VGG16']):
    ax = axes[ax_idx]
    for fold_idx in range(5):
        h2 = all_histories[model][fold_idx]['phase2']
        epochs = range(1, len(h2['val_accuracy']) + 1)
        ax.plot(epochs, h2['val_accuracy'], alpha=0.4,
                label=f'Podział {fold_idx+1}')

    max_len = max(len(all_histories[model][f]['phase2']['val_accuracy']) for f in range(5))
    means = []
    for ep in range(max_len):
        vals = [all_histories[model][f]['phase2']['val_accuracy'][ep]
                for f in range(5)
                if ep < len(all_histories[model][f]['phase2']['val_accuracy'])]
        means.append(np.mean(vals))
    ax.plot(range(1, max_len + 1), means, 'k-', linewidth=2.5, label='Średnia')

    ax.set_title(f'{model}: dokładność walidacyjna (faza 2)')
    ax.set_xlabel('Epoka')
    ax.set_ylabel('Dokładność walidacyjna')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig2_learning_curves.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# === UŚREDNIONE MACIERZE POMYŁEK ===
for model_name, cmap in [('ResNet50', 'Blues'), ('VGG16', 'Oranges')]:
    cm_mean = np.mean(np.stack(all_cms[model_name]), axis=0)

    plt.figure(figsize=(14, 12))
    sns.heatmap(cm_mean, annot=True, fmt='.1f', cmap=cmap,
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Średnia liczba próbek z 5 podziałów'})
    plt.title(f'Uśredniona macierz pomyłek: {model_name} (walidacja krzyżowa z 5 podziałami)')
    plt.ylabel('Klasa rzeczywista')
    plt.xlabel('Klasa przewidywana')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    fname = 'fig3' if model_name == 'ResNet50' else 'fig4'
    plt.savefig(f'{OUTPUT_DIR}/{fname}_cm_{model_name.lower()}.png',
                dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:
# === GRAD-CAM ===
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg
import matplotlib.cm as cm

gradcam_fold_dir = fold_dirs[FOLD_FOR_GRADCAM]
resnet_weights = f'{gradcam_fold_dir}/resnet50_best.h5'
vgg_weights = f'{gradcam_fold_dir}/vgg16_best.h5'

if not (os.path.exists(resnet_weights) and os.path.exists(vgg_weights)):
    print(f'Brak wag dla podziału {FOLD_FOR_GRADCAM+1}.')
    print('Upewnij się że w treningowym notebooku ustawiono SAVE_WEIGHTS=True')
    model_resnet = None
    model_vgg = None
else:
    print(f'Wczytuję wagi z podziału {FOLD_FOR_GRADCAM+1}')
    model_resnet = load_model(resnet_weights)
    model_vgg = load_model(vgg_weights)


In [ ]:
# === GENEROWANIE MAP GRAD-CAM ===
def make_gradcam(img_array, seq_model, conv_layer_name):
    backbone = seq_model.layers[0]
    inputs = backbone.input
    conv_out = backbone.get_layer(conv_layer_name).output
    x = backbone.output
    for layer in seq_model.layers[1:]:
        x = layer(x)
    grad_model = tf.keras.Model(inputs=inputs, outputs=[conv_out, x])

    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_array)
        pred_idx = tf.argmax(preds[0])
        loss = preds[:, pred_idx]

    grads = tape.gradient(loss, conv_outputs)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_mean(tf.multiply(pooled, conv_outputs[0]), axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

sample_classes = ['Tomato_Bacterial_spot', 'Tomato_Late_blight',
                   'Tomato_healthy', 'Potato___Early_blight']

if model_resnet is not None and model_vgg is not None:
    jet = plt.colormaps['jet']
    fig, axes = plt.subplots(len(sample_classes), 3, figsize=(15, 5 * len(sample_classes)))

    for idx, cls in enumerate(sample_classes):
        cls_path = os.path.join(SRC, cls)
        if not os.path.exists(cls_path):
            print(f'Pomijam klasę {cls}: brak ścieżki')
            continue
        img_path = os.path.join(cls_path, sorted(os.listdir(cls_path))[0])
        orig = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
        orig_arr = tf.keras.preprocessing.image.img_to_array(orig)

        axes[idx, 0].imshow(orig)
        axes[idx, 0].set_title(f'Oryginał\n{cls}', fontsize=9)
        axes[idx, 0].axis('off')

        for col, (mdl, preproc, layer_name, mname) in enumerate([
            (model_resnet, preprocess_resnet, 'conv5_block3_out', 'ResNet50'),
            (model_vgg, preprocess_vgg, 'block5_conv3', 'VGG16'),
        ], start=1):
            img_in = preproc(np.expand_dims(orig_arr.copy(), axis=0))
            heatmap = make_gradcam(img_in, mdl, layer_name)
            heatmap = tf.image.resize(heatmap[..., None], (224, 224)).numpy()[..., 0]
            jet_colors = jet(heatmap)[..., :3]
            overlay = 0.5 * (orig_arr / 255.0) + 0.5 * jet_colors

            axes[idx, col].imshow(overlay)
            axes[idx, col].set_title(f'{mname} Grad-CAM\n(podział {FOLD_FOR_GRADCAM+1})', fontsize=9)
            axes[idx, col].axis('off')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig5_gradcam.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('Pominięto Grad-CAM, brak modeli.')


In [ ]:
# === LICZBA PARAMETRÓW ===
if model_resnet is not None and model_vgg is not None:
    def count_trainable(model):
        return int(sum(tf.size(w).numpy() for w in model.trainable_weights))

    n_resnet = count_trainable(model_resnet)
    n_vgg = count_trainable(model_vgg)
    print(f'ResNet50 parametry trenowalne (uczenie transferowe): {n_resnet:>12,}')
    print(f'VGG16    parametry trenowalne (uczenie transferowe): {n_vgg:>12,}')
    print(f'\nW milionach: ResNet50 ok. {n_resnet/1e6:.2f} mln, VGG16 ok. {n_vgg/1e6:.2f} mln')


In [ ]:
# === EKSPORT ZBIORCZY ===
final_summary = {
    'experiment': 'stratyfikowana walidacja krzyżowa z 5 podziałami',
    'random_seed': 42,
    'n_folds': 5,
    'models': {},
    'wilcoxon_tests': {},
}

for model in ['ResNet50', 'VGG16']:
    model_data = {
        'per_fold': [r for r in all_results[model]],
        'aggregate': {},
    }
    for metric in metrics_to_report:
        vals = np.array([r[metric] for r in all_results[model]])
        model_data['aggregate'][metric] = {
            'mean': float(vals.mean()),
            'std': float(vals.std()),
            'min': float(vals.min()),
            'max': float(vals.max()),
            'values': [float(v) for v in vals],
        }
    final_summary['models'][model] = model_data

for metric in metrics_to_report:
    resnet_vals = np.array([r[metric] for r in all_results['ResNet50']])
    vgg_vals = np.array([r[metric] for r in all_results['VGG16']])
    if not np.allclose(resnet_vals, vgg_vals):
        stat, p = wilcoxon(resnet_vals, vgg_vals)
        final_summary['wilcoxon_tests'][metric] = {
            'statistic': float(stat),
            'p_value': float(p),
            'significant_at_0.05': bool(p < 0.05),
        }

with open(f'{OUTPUT_DIR}/final_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=2)

print(f'Wszystkie wyniki zapisane do {OUTPUT_DIR}/')
import subprocess
print()
print(subprocess.run(['ls', '-lh', OUTPUT_DIR], capture_output=True, text=True).stdout)


In [ ]:
# === ROZSZERZONY PRZEGLĄD GRAD-CAM: 2 OBRAZY × 15 KLAS ==========================

import os
import csv
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

RANDOM_SEED = 42
N_FOLDS = 5

if model_resnet is None or model_vgg is None:
    raise RuntimeError('Brak modeli — uruchom najpierw komórki Grad-CAM powyżej.')

# === ODTWORZENIE PODZIAŁU TESTOWEGO DLA WYBRANEGO FOLDA =================
rows = []
for cls in sorted(os.listdir(SRC)):
    cls_path = os.path.join(SRC, cls)
    if not os.path.isdir(cls_path):
        continue
    for f in sorted(os.listdir(cls_path)):
        rows.append({'filepath': os.path.join(cls_path, f), 'class': cls})
df = pd.DataFrame(rows)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_splits = list(skf.split(df, df['class']))
_, test_idx = fold_splits[FOLD_FOR_GRADCAM]
df_test = df.iloc[test_idx].reset_index(drop=True)

print(f'Zbiór testowy podziału {FOLD_FOR_GRADCAM+1}: {len(df_test)} obrazów')

# === WYBÓR 2 OBRAZÓW NA KLASĘ ========
classes = sorted(df_test['class'].unique())
print(f'Klas w zbiorze testowym: {len(classes)}')

selected = []
for cls in classes:
    cls_files = sorted(df_test[df_test['class'] == cls]['filepath'].tolist())
    if len(cls_files) < 2:
        print(f'  UWAGA: klasa {cls} ma tylko {len(cls_files)} obrazów testowych')
    selected.append({'cls': cls, 'paths': cls_files[:2]})

# === KONTAKTOWE ARKUSZE PER KLASA + KOLEKCJONOWANIE WIERSZY CSV =========
review_dir = f'{OUTPUT_DIR}/gradcam_review'
os.makedirs(review_dir, exist_ok=True)

jet = plt.colormaps['jet']
csv_rows = []

for cls_idx, item in enumerate(selected, start=1):
    cls = item['cls']
    paths = item['paths']
    n_rows = len(paths)
    if n_rows == 0:
        continue

    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for r, img_path in enumerate(paths):
        orig = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
        orig_arr = tf.keras.preprocessing.image.img_to_array(orig)

        axes[r, 0].imshow(orig)
        axes[r, 0].set_title(f'Oryginał\n{os.path.basename(img_path)}', fontsize=9)
        axes[r, 0].axis('off')

        for col, (mdl, preproc, layer_name, mname) in enumerate([
            (model_resnet, preprocess_resnet, 'conv5_block3_out', 'ResNet50'),
            (model_vgg,    preprocess_vgg,    'block5_conv3',     'VGG16'),
        ], start=1):
            img_in = preproc(np.expand_dims(orig_arr.copy(), axis=0))
            heatmap = make_gradcam(img_in, mdl, layer_name)
            heatmap = tf.image.resize(heatmap[..., None], (224, 224)).numpy()[..., 0]
            jet_colors = jet(heatmap)[..., :3]
            overlay = 0.5 * (orig_arr / 255.0) + 0.5 * jet_colors

            axes[r, col].imshow(overlay)
            axes[r, col].set_title(f'{mname} Grad-CAM\n(podział {FOLD_FOR_GRADCAM+1})', fontsize=9)
            axes[r, col].axis('off')

        csv_rows.append({
            'klasa': cls,
            'obraz': os.path.basename(img_path),
            'kategoria_resnet50': '',  # do wypełnienia: A / B / C / D
            'kategoria_vgg16':    '',  # do wypełnienia: A / B / C / D
            'uwagi':              '',  # opcjonalne notatki
        })

    plt.suptitle(f'Klasa: {cls}', fontsize=14, y=1.00)
    plt.tight_layout()
    out_path = f'{review_dir}/{cls_idx:02d}_{cls}.png'
    plt.savefig(out_path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    print(f'  zapisano: {out_path}')

# === TEMPLATE CSV DO RĘCZNEGO KATALOGOWANIA ============================
csv_path = f'{review_dir}/_katalogowanie.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(
        f, fieldnames=['klasa', 'obraz', 'kategoria_resnet50', 'kategoria_vgg16', 'uwagi']
    )
    writer.writeheader()
    for row in csv_rows:
        writer.writerow(row)

print(f'\nTemplate katalogowania: {csv_path}')
print(f'Map do oceny: {len(csv_rows)} obrazów × 2 modele = {len(csv_rows)*2} map.')
print()
print('Kategorie:')
print('  A — aktywacja skupiona na zmianie chorobowej')
print('      (dla klas zdrowych: aktywacja na obszarze liścia)')
print('  B — aktywacja rozproszona po liściu (bez wyraźnego skupienia)')
print('  C — aktywacja głównie na tle / poza obszarem liścia')
print('  D — brak czytelnej aktywacji (mapa głównie ciemnoniebieska)')

In [ ]:
# === REGENERACJA I REPREZENTATYWNE PRZYKŁADY Z PRZEGLĄDU =================

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

if model_resnet is None or model_vgg is None:
    raise RuntimeError('Brak modeli — uruchom najpierw komórki Grad-CAM powyżej.')

selected_images = [
    ('Tomato__Target_Spot',
     '003a5321-0430-42dd-a38d-30ac4563f4ba___Com.G_TgS_FL 8121.JPG'),
    ('Pepper__bell___Bacterial_spot',
     '04d46cfb-9cc8-4083-82af-ca2bb57c8182___NREC_B.Spot 1814.JPG'),
    ('Potato___healthy',
     '04481ca2-f94c-457e-b785-1ac05800b7ec___RS_HL 1930.JPG'),
    ('Tomato_Spider_mites_Two_spotted_spider_mite',
     '0100e504-f6bc-464a-a181-38262117f30a___Com.G_SpM_FL 1202.JPG'),
]

# Walidacja obecności plików przed rysowaniem
for cls, fname in selected_images:
    path = os.path.join(SRC, cls, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Brak pliku: {path}')

jet = plt.colormaps['jet']
fig, axes = plt.subplots(len(selected_images), 3,
                          figsize=(15, 5 * len(selected_images)))

for idx, (cls, fname) in enumerate(selected_images):
    img_path = os.path.join(SRC, cls, fname)
    orig = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    orig_arr = tf.keras.preprocessing.image.img_to_array(orig)

    axes[idx, 0].imshow(orig)
    axes[idx, 0].set_title(f'Oryginał\n{cls}', fontsize=9)
    axes[idx, 0].axis('off')

    for col, (mdl, preproc, layer_name, mname) in enumerate([
        (model_resnet, preprocess_resnet, 'conv5_block3_out', 'ResNet50'),
        (model_vgg,    preprocess_vgg,    'block5_conv3',     'VGG16'),
    ], start=1):
        img_in = preproc(np.expand_dims(orig_arr.copy(), axis=0))
        heatmap = make_gradcam(img_in, mdl, layer_name)
        heatmap = tf.image.resize(heatmap[..., None], (224, 224)).numpy()[..., 0]
        jet_colors = jet(heatmap)[..., :3]
        overlay = 0.5 * (orig_arr / 255.0) + 0.5 * jet_colors

        axes[idx, col].imshow(overlay)
        axes[idx, col].set_title(f'{mname} Grad-CAM\n(podział {FOLD_FOR_GRADCAM+1})',
                                  fontsize=9)
        axes[idx, col].axis('off')

plt.tight_layout()
out_path = f'{OUTPUT_DIR}/fig5_gradcam.png'
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Zapisano: {out_path}')